In [2]:
from pyspark.sql import SparkSession
import os

home = os.path.expanduser("~")

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("TaaSim-MinIO-Test") \
    .config("spark.jars", 
            f"{home}/hadoop-aws-3.3.4.jar,{home}/aws-java-sdk-bundle-1.12.262.jar") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "taasim") \
    .config("spark.hadoop.fs.s3a.secret.key", "taasim123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .getOrCreate()

In [3]:
# Lire UNE seule partition pour vérifier
df = spark.read.parquet("s3a://curated/trips/year_month=2013-07/")

print(f"Nombre de lignes : {df.count()}")
print(f"\nColonnes disponibles :")
df.printSchema()

Nombre de lignes : 103617

Colonnes disponibles :
root
 |-- trip_id: string (nullable = true)
 |-- taxi_id: integer (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- snapped_polyline: string (nullable = true)
 |-- origin_zone_id: integer (nullable = true)
 |-- origin_zone_name: string (nullable = true)
 |-- dest_zone_id: integer (nullable = true)
 |-- dest_zone_name: string (nullable = true)
 |-- trip_duration_sec: integer (nullable = true)



In [4]:
# Voir 5 lignes concrètes
df.show(5, truncate=False)

+-------------------+--------+----------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [5]:
# Afficher uniquement les noms de colonnes
print(df.columns)

['trip_id', 'taxi_id', 'timestamp', 'snapped_polyline', 'origin_zone_id', 'origin_zone_name', 'dest_zone_id', 'dest_zone_name', 'trip_duration_sec']


In [6]:
import pandas as pd

sample = df.select("timestamp").limit(3).toPandas()
print(sample)

# Convertir pour vérifier
ts = sample["timestamp"][0]
print(pd.to_datetime(ts, unit='s'))  # doit afficher une date 2013-2014

    timestamp
0  1372641041
1  1372641149
2  1372662598
2013-07-01 01:10:41


In [7]:
df.select("origin_zone_id").distinct().orderBy("origin_zone_id").show()
# doit afficher 0 à 16

+--------------+
|origin_zone_id|
+--------------+
|             0|
|             1|
|             2|
|             3|
|             4|
|             5|
|             6|
|             7|
|             8|
|             9|
|            10|
|            11|
|            12|
|            13|
|            14|
|            15|
|            16|
+--------------+

